# VGG网络

VGG网络类似于AlexNet网络，只不过他将卷积层和池化层封装成块，最后几个块串联之后连接两个全连接层进行输出

下面用代码定义VGG11网络，观察其各层的设置

下面的代码定义了一个block的卷积层数量和输入输出通道数。

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l


def vgg_block(num_convs, in_channels, out_channels):
    layers = []
    for _ in range(num_convs):
        layers.append(nn.Conv2d(in_channels, out_channels,
                                kernel_size=3, padding=1))  # 3*3kernel，padding=1，保持输入输出尺寸相同
        layers.append(nn.ReLU())  # 卷积层之后添加ReLU激活函数
        in_channels = out_channels  # 第一个卷积层的输出通道数，作为第二个卷积层的输入通道数
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2))  # 添加池化层
    return nn.Sequential(*layers)

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


原始的VGG有五个卷积块，其中前两个块各有一个卷积层，后三个块各包含两个卷积层。 第一个模块有64个输出通道，每个后续模块将输出通道数量翻倍，直到该数字达到512。由于该网络使用8个卷积层和3个全连接层，因此它通常被称为VGG-11

In [ ]:
conv_arch = ((1, 64), (1, 128), (2, 256), (2, 512), (2, 512))  # （卷积层数量，输出通道数）


下面可以通过一个for循环去实现简单的VGG的网络

In [ ]:
def vgg(conv_arch):
    conv_blks = []
    in_channels = 1
    # 卷积层部分
    for (num_convs, out_channels) in conv_arch:
        conv_blks.append(vgg_block(num_convs, in_channels, out_channels))
        in_channels = out_channels

    return nn.Sequential(
        *conv_blks, nn.Flatten(),
        # 全连接层部分
        nn.Linear(out_channels * 7 * 7, 4096), nn.ReLU(), nn.Dropout(0.5),  # 需要在一个全连接层后面添加激活函数和Dropout层
        nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(4096, 10))

net = vgg(conv_arch)

观察一下每个层的输出情况：

In [4]:
X = torch.randn(size=(1, 1, 224, 224))
for blk in net:
    X = blk(X)
    print(blk.__class__.__name__,'output shape:\t',X.shape)

Sequential output shape:	 torch.Size([1, 64, 112, 112])
Sequential output shape:	 torch.Size([1, 128, 56, 56])
Sequential output shape:	 torch.Size([1, 256, 28, 28])
Sequential output shape:	 torch.Size([1, 512, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
Flatten output shape:	 torch.Size([1, 25088])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 10])


数据的流转如下所示：

输入：224×224
  ↓ 卷积块1 + 池化

   112×112
  ↓ 卷积块2 + 池化

   56×56
  ↓ 卷积块3 + 池化

   28×28
  ↓ 卷积块4 + 池化

   14×14
  ↓ 卷积块5 + 池化

   7×7  ← 最终特征图尺寸！